# Full GUDS OOD & Fairness Adaptation on PAD-UFES-20 (NVIDIA-layout)

This notebook trains a source-only control and a target-assisted OOD model. It then performs three evaluations:
1. **Unadapted OOD Detection** on the held-out `imgs_part_3` partition.
2. **Frozen-backbone Adaptation** on `partition=all` to compute leakage-safe out-of-fold generalization and skin-tone/gender fairness (Equalized Odds Gap).
3. **Layer-4 KD Adaptation** on `partition=all` to adapt features under distillation constraints.

In [ ]:
# Cell 1 -- pull code; check requirements and locate datasets
import os, sys, json, shutil, subprocess, shlex
from pathlib import Path

REPO_URL = 'https://github.com/minhduc110207/MDEP-Microglial-Driven-Evidential-Pruning.git'
REPO_BRANCH = 'main'
REPO_ROOT = Path('/kaggle/working/MDEP-Microglial-Driven-Evidential-Pruning')
WORK, OUT = Path('/kaggle/working'), Path('/kaggle/working/paper_experiment_outputs')
OUT.mkdir(parents=True, exist_ok=True)

def run(cmd, cwd=None, env=None):
    cmd = [str(x) for x in cmd]
    print('\n$', shlex.join(cmd), flush=True)
    merged = os.environ.copy(); merged.update({str(k): str(v) for k, v in (env or {}).items()})
    subprocess.run(cmd, cwd=str(cwd or REPO_ROOT), env=merged, check=True)

if not (REPO_ROOT / '.git').exists():
    run(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, REPO_ROOT], cwd=WORK)
else:
    run(['git', 'fetch', 'origin', REPO_BRANCH], cwd=REPO_ROOT)
    run(['git', 'checkout', REPO_BRANCH], cwd=REPO_ROOT)
    run(['git', 'pull', '--ff-only', 'origin', REPO_BRANCH], cwd=REPO_ROOT)

def find_isic(root):
    root = Path(root)
    for csv in [root / 'train-metadata.csv', *root.rglob('train-metadata.csv')]:
        p = csv.parent
        if (p / 'train-image.hdf5').exists() or (p / 'train-image' / 'image').exists(): return p
    return None

def find_pad(root):
    root = Path(root)
    for csv in [root / 'metadata.csv', *root.rglob('metadata.csv')]:
        p = csv.parent
        if all((p / x).exists() for x in ('imgs_part_1', 'imgs_part_2', 'imgs_part_3')): return p
    return None

ISIC_ROOT = next((p for p in (find_isic(x) for x in [Path('/kaggle/input/competitions/isic-2024-challenge'), Path('/kaggle/input/isic-2024-challenge')]) if p), None)
PAD_ROOT = next((p for p in (find_pad(x) for x in [Path('/kaggle/input/datasets/mahdavi1202/skin-cancer'), Path('/kaggle/input/skin-cancer')]) if p), None)
if ISIC_ROOT is None or PAD_ROOT is None:
    try:
        import kagglehub
        if ISIC_ROOT is None: ISIC_ROOT = find_isic(kagglehub.competition_download('isic-2024-challenge'))
        if PAD_ROOT is None: PAD_ROOT = find_pad(kagglehub.dataset_download('mahdavi1202/skin-cancer'))
    except Exception as exc:
        raise RuntimeError('Attach ISIC-2024 and PAD-UFES-20 through Add Data (or enable Internet and accept the ISIC competition rules).') from exc
assert ISIC_ROOT is not None, 'ISIC train-metadata/train images not found'
assert PAD_ROOT is not None, 'PAD metadata plus imgs_part_1/2/3 not found'
PAD_META = PAD_ROOT / 'metadata.csv'
assert PAD_META.exists()
os.environ['ISIC_ROOT'] = str(ISIC_ROOT)
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_ROOT, text=True).strip()
print('[OK] commit =', commit)
print('[OK] ISIC_ROOT =', ISIC_ROOT)
print('[OK] PAD_ROOT =', PAD_ROOT, '| OE=imgs_part_1/2, test=imgs_part_3')


In [ ]:
# Cell 2 -- fixed, leakage-safe training. Do not tune on imgs_part_3.
SEEDS = (42, 123, 456)
SPLIT_SEED, EPOCHS, BATCH_SIZE, LR = 42, 40, 32, 4e-5
PROFILE = 'nvidia_v3'
VARIANTS = {
    # Clean external-OOD control: no PAD image is seen while training.
    'source_only': {'suffix': '_ood_padufes_source_only_v1', 'oe': None, 'primary': 'knn_layer3'},
    # Fixed before testing: PAD parts 1/2 only, detached head protects ISIC classifier/topology.
    'target_assisted_projection': {'suffix': '_ood_padufes_projection_v1', 'oe': str(PAD_ROOT), 'primary': 'knn_ood_projection'},
}
OOD_OUT = OUT / 'ood_padufes'; OOD_OUT.mkdir(parents=True, exist_ok=True)

def model_metrics(variant, seed):
    return OUT / 'isic' / f"full_guds{VARIANTS[variant]['suffix']}" / f'seed_{seed}' / 'metrics.json'

def train_complete(variant, seed):
    try:
        d = json.loads(model_metrics(variant, seed).read_text())
        return (d.get('protocol_profile') == PROFILE and d.get('sparsity_layout') == 'nvidia_kcrs' and
                d.get('epochs') == EPOCHS and d.get('model_seed', d.get('seed')) == seed and
                d.get('outlier_exposure') == VARIANTS[variant]['oe'] and
                d.get('ood_loss_target') == 'projection' and d.get('checkpoint_selection', {}).get('metric') == 'last')
    except (OSError, ValueError, TypeError): return False

for variant, cfg in VARIANTS.items():
    for seed in SEEDS:
        if train_complete(variant, seed):
            print(f'[SKIP] train {variant}, seed={seed}')
            continue
        cmd = [sys.executable, '-u', 'experiments/isic_paper_experiments.py', '--experiment', 'full_guds',
               '--seed', str(seed), '--split_seed', str(SPLIT_SEED), '--epochs', str(EPOCHS),
               '--batch_size', str(BATCH_SIZE), '--lr', str(LR), '--subsample_scope', 'train',
               '--subsample_ratio', '20', '--checkpoint_selection', 'last',
               '--structural_proxy_batches', '4', '--protocol_profile', PROFILE,
               '--run_suffix', cfg['suffix'], '--ood_loss_target', 'projection',
               '--lambda_ood', '0.003', '--ood_batch_ratio', '0.125', '--ood_start_epoch', '30']
        if cfg['oe'] is not None:
            cmd += ['--outlier_exposure', cfg['oe']]
        run(cmd, cwd=REPO_ROOT, env={'ISIC_ROOT': ISIC_ROOT, 'MDEP_DETERMINISTIC': '1'})

assert all(train_complete(v, s) for v in VARIANTS for s in SEEDS), 'A train run failed or did not meet the NVIDIA OOD protocol.'
print('[DONE] All Full GUDS OOD checkpoints passed the NVIDIA-layout gate.')


In [ ]:
# Cell 3 -- OOD evaluation on held-out PAD imgs_part_3
def audit_path(variant, seed): return OOD_OUT / f'{variant}_seed_{seed}_imgs_part_3.json'

def eval_complete(variant, seed):
    try:
        d = json.loads(audit_path(variant, seed).read_text())
        r = d['external_summary']
        return (d['variant'] == variant and r['checkpoint_protocol']['nvidia_layout_validated'] and
                r['external_dataset']['pad_ufes_partition'] == 'imgs_part_3' and
                r['primary_ood']['score'] == VARIANTS[variant]['primary'])
    except (OSError, KeyError, ValueError, TypeError): return False

for variant, cfg in VARIANTS.items():
    for seed in SEEDS:
        if eval_complete(variant, seed):
            print(f'[SKIP] held-out OOD {variant}, seed={seed}')
            continue
        checkpoint = model_metrics(variant, seed).with_name('model_state.pth')
        run([sys.executable, '-u', 'experiments/run_external_validation.py',
             '--model_path', checkpoint, '--seed', str(seed), '--split_seed', str(SPLIT_SEED),
             '--custom_image_folder', PAD_ROOT, '--pad_ufes_csv', PAD_META,
             '--pad_ufes_partition', 'imgs_part_3', '--knn_primary_layer', 'layer3',
             '--primary_ood_score', cfg['primary'], '--fairness_min_group_size', '20',
             '--fairness_min_class_size', '10'], cwd=REPO_ROOT, env={'ISIC_ROOT': ISIC_ROOT, 'MDEP_DETERMINISTIC': '1'})
        candidates = list((OUT / 'external_validation').glob(f'external_validation_seed_{seed}_imgs_part_3_*.json'))
        assert candidates, f'No external-validation JSON emitted for seed={seed}'
        raw = max(candidates, key=lambda p: p.stat().st_mtime)
        wrapped = {'variant': variant, 'checkpoint': str(checkpoint), 'git_commit': commit,
                   'target_assisted': cfg['oe'] is not None, 'external_summary': json.loads(raw.read_text())}
        audit_path(variant, seed).write_text(json.dumps(wrapped, indent=2))

assert all(eval_complete(v, s) for v in VARIANTS for s in SEEDS), 'OOD evaluation incomplete.'
print('[DONE] PAD imgs_part_3 remained held out and all seed summaries were preserved.')


In [ ]:
# Cell 4 -- Domain Adaptation on entire PAD-UFES-20 (partition=all) to get stable Fairness metrics
def adaptation_complete(variant):
    return (OUT / 'pad_adaptation' / variant / 'pad_adaptation_summary.json').exists()

for variant, cfg in VARIANTS.items():
    v_out = OUT / 'pad_adaptation' / variant
    if adaptation_complete(variant):
        print(f'[SKIP] adaptation for {variant}')
        continue
    ckpt_tmpl = str(OUT / 'isic' / f"full_guds{cfg['suffix']}" / "seed_{seed}" / "model_state.pth")
    cmd = [sys.executable, '-u', 'experiments/run_pad_adaptation.py',
           '--pad_root', PAD_ROOT, '--pad_csv', PAD_META, '--partition', 'all',
           '--model_path', ckpt_tmpl, '--seeds', *map(str, SEEDS), '--split_seed', str(SPLIT_SEED),
           '--target_mode', 'diagnosis6', '--feature_layer', 'auto', '--head', 'mlp', # Using MLP to maximize metrics
           '--outer_folds', '5', '--inner_folds', '3',
           '--fairness_min_group_size', '20', '--fairness_min_class_size', '10',
           '--target_sensitivity', '0.80', '--bootstrap_repeats', '200',
           '--batch_size', '32', '--num_workers', '2',
           '--output_dir', str(v_out)]
    if variant == 'target_assisted_projection':
        cmd.append('--train_domain_head')
    run(cmd, cwd=REPO_ROOT, env={'ISIC_ROOT': ISIC_ROOT, 'MDEP_DETERMINISTIC': '1'})

assert all(adaptation_complete(v) for v in VARIANTS), 'PAD adaptation incomplete.'
print('[DONE] PAD frozen-feature nested adaptation completed.')


In [ ]:
# Cell 5 -- Layer-4 adaptation with Knowledge Distillation (run_pad_layer4_kd.py)
def kd_complete(variant):
    return (OUT / 'pad_layer4_kd' / variant / 'pad_layer4_kd_summary.json').exists()

for variant, cfg in VARIANTS.items():
    v_out = OUT / 'pad_layer4_kd' / variant
    if kd_complete(variant):
        print(f'[SKIP] layer4 kd for {variant}')
        continue
    ckpt_tmpl = str(OUT / 'isic' / f"full_guds{cfg['suffix']}" / "seed_{seed}" / "model_state.pth")
    cmd = [sys.executable, '-u', 'experiments/run_pad_layer4_kd.py',
           '--pad_root', PAD_ROOT, '--pad_csv', PAD_META, '--partition', 'all',
           '--model_path', ckpt_tmpl, '--seeds', *map(str, SEEDS), '--split_seed', str(SPLIT_SEED),
           '--target_mode', 'diagnosis6', '--outer_folds', '5', '--inner_folds', '3',
           '--epochs', '12', '--lr', '1e-5', '--kd_weight', '2.0',
           '--kd_temperature', '2.0', '--target_sensitivity', '0.80',
           '--bootstrap_repeats', '200', '--batch_size', '32', '--num_workers', '2',
           '--output_dir', str(v_out)]
    run(cmd, cwd=REPO_ROOT, env={'ISIC_ROOT': ISIC_ROOT, 'MDEP_DETERMINISTIC': '1'})

assert all(kd_complete(v) for v in VARIANTS), 'Layer-4 KD adaptation incomplete.'
print('[DONE] PAD Layer-4 KD nested adaptation completed.')


In [ ]:
# Cell 6 -- Report all results (OOD Detection, Generalization, and Fairness)
import pandas as pd

print('================================================================================')
print('1. UNADAPTED OOD DETECTION METRICS (on held-out imgs_part_3)')
print('================================================================================')
rows_ood = []
for variant in VARIANTS:
    for seed in SEEDS:
        d = json.loads(audit_path(variant, seed).read_text())['external_summary']
        p = d['primary_ood']['result']
        rows_ood.append({
            'variant': variant, 'seed': seed,
            'primary_score': d['primary_ood']['score'],
            'full_auroc': p['full_auroc'], 'full_aupr': p['full_aupr'],
            'balanced_auroc': p['bal_auroc_mean'], 'balanced_aupr': p['bal_aupr_mean']
        })
df_ood = pd.DataFrame(rows_ood)
display(df_ood.groupby(['variant', 'primary_score'])[['full_auroc', 'balanced_auroc', 'balanced_aupr']].agg(['mean', 'std']))

print('\n================================================================================')
print('2. GENERALIZATION & FAIRNESS VIA FROZEN-BACKBONE ADAPTATION (on partition=all)')
print('================================================================================')
rows_adapt = []
for variant in VARIANTS:
    p_summary = OUT / 'pad_adaptation' / variant / 'pad_adaptation_summary.json'
    if p_summary.exists():
        d = json.loads(p_summary.read_text())
        oof = d['overall_oof']
        fair = d['overall_fairness']
        
        # Extract EOM gap for skin type (fitspatrick) and gender
        fitz_gap = fair.get('fitspatrick', {}).get('fold_specific_group_thresholds', {}).get('eom_gap', 'N/A')
        gender_gap = fair.get('gender', {}).get('fold_specific_group_thresholds', {}).get('eom_gap', 'N/A')
        
        rows_adapt.append({
            'variant': variant,
            'oof_auroc': oof['auroc'], 'oof_ap': oof['average_precision'],
            'oof_balanced_acc': oof['balanced_accuracy'],
            'fitzpatrick_eom_gap': fitz_gap,
            'gender_eom_gap': gender_gap
        })
df_adapt = pd.DataFrame(rows_adapt)
display(df_adapt)

print('\n================================================================================')
print('3. GENERALIZATION & AUDIT VIA LAYER-4 KD ADAPTATION (on partition=all)')
print('================================================================================')
rows_kd = []
for variant in VARIANTS:
    p_summary = OUT / 'pad_layer4_kd' / variant / 'pad_layer4_kd_summary.json'
    if p_summary.exists():
        d = json.loads(p_summary.read_text())
        oof = d['overall_oof']
        rows_kd.append({
            'variant': variant,
            'oof_auroc': oof['auroc'], 'oof_ap': oof['average_precision'],
            'oof_balanced_acc': oof['balanced_accuracy']
        })
df_kd = pd.DataFrame(rows_kd)
display(df_kd)
